# Дружелюбный построитель тепловых карт

Этот ноутбук запускает `heatmap.py` без правки Python-кода. Загрузите скрипт и свои CSV/XLSX файлы, выберите формат данных ниже и запустите ячейки построения.

In [ ]:
%pip -q install pandas numpy matplotlib openpyxl

## 1. Загрузите файлы

Загрузите `heatmap.py` из этой папки и файлы эксперимента, по которым нужно построить тепловые карты. Можно загружать CSV, TSV, XLS и XLSX.

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
    print("Загружено:", ", ".join(uploaded.keys()) or "ничего")
except Exception:
    print("Кнопка загрузки работает в Google Colab. При локальном запуске положите файлы рядом с ноутбуком.")

print("\nФайлы в текущей среде:")
for path in sorted(Path(".").glob("*")):
    if path.is_file():
        print(" -", path.name)

## 2. Настройте данные

Для большинства новых экспериментов основной вариант - Excel с тройками `mean / SD / N`. Если данные уже собраны иначе, используйте длинную таблицу или широкую таблицу с дозами в колонках `3_Gy`, `6_Gy`, `9_Gy`.

In [ ]:
#@title Файлы и формат данных
mode = "текущий пример: два Excel pH + ручной CSV" #@param ["текущий пример: два Excel pH + ручной CSV", "одна длинная таблица", "одна широкая таблица с дозами", "Excel-файлы списком file=pH"]

excel_ph_65 = "Protons BiCe GdEuF3 6_5.xlsx" #@param {type:"string"}
ph_65 = 6.5 #@param {type:"number"}
excel_ph_74 = "Protons BiCe GdEuF3 7_4.xlsx" #@param {type:"string"}
ph_74 = 7.4 #@param {type:"number"}
manual_wide_csv = "gdf3_gdeuf3_manual.csv" #@param {type:"string"}

single_table_file = "" #@param {type:"string"}
single_table_family = "All samples" #@param {type:"string"}
excel_files_with_ph = "" #@param {type:"string"}

# Примеры: BiCe=BiCe; GdEu=GdF3 / GdEuF3; HeLa=HeLa cells
family_rules = "BiCe=BiCe; GdEu=GdF3 / GdEuF3; GdF3=GdF3 / GdEuF3" #@param {type:"string"}
default_family = "All samples" #@param {type:"string"}

#@title Внешний вид графика
output_dir = "heatmap_outputs" #@param {type:"string"}
value_label = "$\\ln(\\mathrm{DEF}_{\\mathrm{ROS}})$" #@param {type:"string"}
dose_label = "Dose, Gy" #@param {type:"string"}
concentration_label = "Conc, mg/mL" #@param {type:"string"}
panel_label = "pH" #@param {type:"string"}
colorbar_note = "blue: ROS decrease vs control; red: ROS increase vs control" #@param {type:"string"}
color_map = "bwr" #@param ["bwr", "coolwarm", "RdBu_r", "viridis", "magma", "plasma"]
center_colors_at_zero = True #@param {type:"boolean"}
use_one_color_scale_for_all_groups = True #@param {type:"boolean"}
show_numbers_in_cells = True #@param {type:"boolean"}


In [ ]:
import importlib
import json
from pathlib import Path

if not Path("heatmap.py").exists():
    raise FileNotFoundError("Перед запуском этой ячейки загрузите heatmap.py в шаге 1.")

import heatmap
importlib.reload(heatmap)

def parse_family_rules(text):
    rules = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Правило группы должно выглядеть так: текст=название группы. Ошибка: {chunk}")
        contains, family = chunk.split("=", 1)
        rules.append({"contains": contains.strip(), "family": family.strip()})
    return rules

def parse_excel_file_list(text):
    sources = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Запись Excel должна выглядеть так: file.xlsx=6.5. Ошибка: {chunk}")
        path, ph = chunk.split("=", 1)
        sources.append({"path": path.strip(), "format": "triplet_excel", "ph": float(ph.strip())})
    return sources

def file_exists(path):
    return bool(path) and Path(path).exists()

sources = []
if mode == "текущий пример: два Excel pH + ручной CSV":
    if file_exists(excel_ph_65):
        sources.append({"path": excel_ph_65, "format": "triplet_excel", "ph": ph_65, "include_families": ["BiCe"]})
    if file_exists(excel_ph_74):
        sources.append({"path": excel_ph_74, "format": "triplet_excel", "ph": ph_74, "include_families": ["BiCe"]})
    if file_exists(manual_wide_csv):
        sources.append({"path": manual_wide_csv, "format": "wide_dose", "family": "GdF3 / GdEuF3"})
elif mode == "одна длинная таблица":
    sources.append({"path": single_table_file, "format": "long", "family": single_table_family})
elif mode == "одна широкая таблица с дозами":
    sources.append({"path": single_table_file, "format": "wide_dose", "family": single_table_family})
elif mode == "Excel-файлы списком file=pH":
    sources.extend(parse_excel_file_list(excel_files_with_ph))

if not sources:
    raise ValueError("Не настроен ни один источник данных. Проверьте имена загруженных файлов и выбранный режим.")

config = {
    "sources": sources,
    "family_rules": parse_family_rules(family_rules),
    "default_family": default_family,
    "plot": {
        "output_dir": output_dir,
        "value_label": value_label,
        "colorbar_note": colorbar_note,
        "dose_label": dose_label,
        "concentration_label": concentration_label,
        "panel_label": panel_label,
        "cmap": color_map,
        "center": 0 if center_colors_at_zero else None,
        "global_color_scale": use_one_color_scale_for_all_groups,
        "annotate": show_numbers_in_cells,
    },
}

print(json.dumps(config, indent=2, ensure_ascii=False))

## 3. Постройте тепловые карты

In [ ]:
from IPython.display import Image, display

df = heatmap.load_dataset(config)
print(f"Загружено строк: {len(df)}")
display(df.head(20))

paths = heatmap.plot_heatmaps(df, config["plot"])
for path in paths:
    print(path)
    display(Image(filename=str(path)))

## 4. Скачайте результаты

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path(output_dir) / "heatmaps.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path, arcname=Path(path).name)

print("Создан архив", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Кнопка скачивания доступна в Google Colab. При локальном запуске откройте ZIP по пути выше.")